In [1]:
from ollama import chat

model="llama3.2:1b"
response = chat(
    model,
    messages=[
        {"role": "system", "content": "You are a helpful AI agent."},
        {"role": "user", "content": "Explain LLMs in one line"}
    ]
)
print(response)

response

The daily Gemini API call limit **is not a single fixed number and can vary** depending on several factors:

1.  **Free Tier (Google AI Studio):** If you are using the Gemini API primarily through Google AI Studio for personal or experimental use, there are **free tier quotas** in place. These limits are generally generous enough for development and learning, but they are designed to prevent abuse. The exact numbers can sometimes fluctuate and may depend on the specific model and region.

2.  **Paid Projects (Google Cloud Vertex AI):** If you are using Gemini as part of a Google Cloud project (Vertex AI), the default quotas are typically **much higher** than the free tier. Furthermore, you can often **request quota increases** through the Google Cloud Console if your application requires more calls.

**How to find your specific limits:**

The most accurate and up-to-date place to check your current Gemini API call limits is directly within the **Google Cloud Console**:

1.  **Go to the

In [2]:
from pydantic import BaseModel, Field

class Conversation(BaseModel):
    question: str = Field(description="question from user")
    ai_response: str = Field(description="LLM response")

In [3]:
from langchain_core.output_parsers import PydanticOutputParser

pydantic_parser = PydanticOutputParser(pydantic_object=Conversation)
pydantic_parser.get_format_instructions()

C:\Users\argroy\arg_venv\Lib\site-packages\langchain_core\_api\deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"question": {"description": "question from user", "title": "Question", "type": "string"}, "ai_response": {"description": "LLM response", "title": "Ai Response", "type": "string"}}, "required": ["question", "ai_response"]}\n```'

In [4]:
llm_output = '''
```json
{
  "question": "What is iron man\'s first movie called?",
  "ai_response": "Iron Man (2008)"
}
```
'''
pydantic_output = pydantic_parser.invoke(llm_output)
print(pydantic_output)

question="What is iron man's first movie called?" ai_response='Iron Man (2008)'


In [11]:
from langchain_core.tracers import LangChainTracer
from langchain_core.runnables import RunnableLambda
# --------------------------------------------------------------
# 0️⃣  Install (run once)
# --------------------------------------------------------------
# pip install --upgrade "langchain[all]" "langchain-ollama"

# --------------------------------------------------------------
# 1️⃣ Imports – new core locations
# --------------------------------------------------------------
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
import json, re, asyncio, sys
from itertools import islice
from langsmith import Client

# --------------------------------------------------------------
# 2️⃣ Pydantic schema + parser
# --------------------------------------------------------------
class MovieInfo(BaseModel):
    """Information about the requested movie."""
    title: str = Field(..., description="Official title of the movie")
    year: int = Field(..., description="Release year (e.g. 2008)")
    director: str = Field(..., description="Full name of the director")

pydantic_parser = PydanticOutputParser(pydantic_object=MovieInfo)

# --------------------------------------------------------------
# 3️⃣ Ollama chat model (JSON‑only mode)
# --------------------------------------------------------------
llm = ChatOllama(
    model="llama3.2:1b",          # any local Ollama model that supports JSON mode
    temperature=0.0,
    max_tokens=1024,
    # **Important** – tell Ollama to emit *pure* JSON, no markdown
    #   The underlying API accepts a `format="json"` kwarg.
    #   In the Python wrapper we pass it via `response_format`.
    response_format="json",     # <-- forces JSON output
)

# --------------------------------------------------------------
# 4️⃣ PromptTemplate – make the instruction crystal‑clear
# --------------------------------------------------------------
# The parser already gives us a block that says “return a JSON object …”.
# We add two more lines that explicitly forbid schema output and markdown.
template = """
You are a helpful assistant that must answer ONLY with a JSON object that matches the schema below.
**Do NOT** output the JSON schema itself, do NOT wrap the answer in markdown fences, and do NOT add any extra commentary.

{format_instructions}

User question: {user_query}
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["user_query"],
    partial_variables={"format_instructions": pydantic_parser.get_format_instructions()},
)

# --------------------------------------------------------------
# 5️⃣ Helper – safe parsing (fallback to regex if needed)
# --------------------------------------------------------------
def safe_parse(text: str) -> MovieInfo:
    """
    Tries the normal Pydantic parser.
    If it fails, extracts the first JSON block with a regex
    and parses that raw dict.
    """
    try:
        return pydantic_parser.invoke(text)
    except Exception as exc:
        # ------------------------------------------------------
        # 5️⃣a  Grab the *first* {...} block (handles markdown fences)
        # ------------------------------------------------------
        json_match = re.search(r"\{[\s\S]*?\}", text)
        if not json_match:
            raise RuntimeError(f"Unable to find JSON in LLM output: {text}") from exc

        candidate = json_match.group(0)
        try:
            data = json.loads(candidate)
            return MovieInfo(**data)
        except Exception as inner:
            raise RuntimeError(
                f"Could not build MovieInfo from extracted JSON: {candidate}"
            ) from inner

# --------------------------------------------------------------
# 6️⃣ METHOD 1 – Classic (prompt → LLM → parse)
# --------------------------------------------------------------
def classic_flow():
    print("\n--- Classic string‑then‑parse flow ---")
    full_prompt = prompt.invoke({"user_query": "What is Iron Man's first movie called?"})
    print("[Prompt sent to Ollama]")
    print(full_prompt)

    # Ollama wrapper accepts a plain string and automatically wraps it into a user message.
    raw_output = llm.invoke(full_prompt)
    print("\n[Raw LLM output]")
    print(raw_output)

    movie = safe_parse(raw_output.content)
    print("\n[Parsed MovieInfo]")
    print(movie)
    print("title:", movie.title, "year:", movie.year, "director:", movie.director)


# --------------------------------------------------------------
# 7️⃣ METHOD 2 – Runnable pipeline (sync + async)
# --------------------------------------------------------------
def runnable_flow():
    print("\n--- Runnable pipeline (sync) ---")
    pipeline = (
        prompt
        | llm
        | StrOutputParser()      # converts AIMessage → plain string
        | RunnableLambda(safe_parse)   # safe_parse returns a MovieInfo
    )
    # optional tracing – prints a tiny tree to stdout
    pipeline = pipeline.with_config(callbacks=[LangChainTracer()])

    result_sync = pipeline.invoke(
        {"user_query": "What is Iron Man's first movie called?"}
    )
    print("\nResult (sync):", result_sync)

    async def async_part():
        result_async = await pipeline.ainvoke(
            {"user_query": "What is Iron Man's first movie called?"}
        )
        print("\nResult (async):", result_async)

    asyncio.run(async_part())


# --------------------------------------------------------------
# 8️⃣ METHOD 3 – Streaming + manual parse
# --------------------------------------------------------------
def streaming_flow():
    print("\n--- Streaming flow (token‑by‑token) ---")
    # Build the prompt string first (the Ollama model expects a string)
    prompt_str = prompt.invoke({"user_query": "What is Iron Man's first movie called?"})

    # Stream the model – we receive AIMessageChunk objects
    token_stream = llm.stream({"messages": [{"role": "user", "content": prompt_str}]})

    collected = ""
    print("[Streaming output]")
    for chunk in token_stream:
        txt = chunk.content
        print(txt, end="", flush=True)   # live display
        collected += txt
    print("\n--- end of stream ---")

    # Parse the final collected text
    movie = safe_parse(collected)
    print("\nParsed after streaming:")
    print(movie)


# --------------------------------------------------------------
# 9️⃣ Run everything
# --------------------------------------------------------------
if __name__ == "__main__":
    # 1️⃣ Classic
    classic_flow()

    # 2️⃣ Runnable pipeline
    runnable_flow()

    # 3️⃣ Streaming
    streaming_flow()



--- Classic string‑then‑parse flow ---
[Prompt sent to Ollama]
text='\nYou are a helpful assistant that must answer ONLY with a JSON object that matches the schema below.\n**Do NOT** output the JSON schema itself, do NOT wrap the answer in markdown fences, and do NOT add any extra commentary.\n\nThe output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"description": "Information about the requested movie.", "properties": {"title": {"description": "Official title of the movie", "title": "Title", "type": "string"}, "year": {"description": "Release year (e.g. 2008)", "title": "Year", "type": "

TypeError: expected string or bytes-like object, got 'AIMessage'

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# Build the pipeline: Prompt → LLM → string → pydantic parser
pipeline = (
    prompt                                     # PromptTemplate is a Runnable
    | llm                                      # ChatOllama (returns an AIMessage)
    | StrOutputParser()                        # AIMessage → plain string
    | pydantic_parser                          # string → MovieInfo (Pydantic)
)

# ---- Synchronous call -------------------------------------------------
result_sync = pipeline.invoke({"user_query": "What is Iron Man's first movie called?"})
print("\n=== Result from Runnable (sync) ===")
print(result_sync)          # => an instance of MovieInfo
print("Title:", result_sync.title)

# ---- Asynchronous call -------------------------------------------------
import asyncio

async def async_demo():
    return await pipeline.ainvoke(
        {"user_query": "What is Iron Man's first movie called?"}
    )

result_async = asyncio.run(async_demo())
print("\n=== Result from Runnable (async) ===")
print(result_async)


In [ ]:
from itertools import islice

def stream_and_parse(user_q: str) -> MovieInfo:
    # 1️⃣ Build the prompt (string) – we still need a string for the LLM
    prompt_str = prompt.invoke({"user_query": user_q})

    # 2️⃣ Stream the LLM
    token_stream = llm.stream({"messages": [{"role": "user", "content": prompt_str}]})

    # 3️⃣ Show the streaming output (optional)
    collected = ""
    print("\n[Streaming response]")
    for chunk in token_stream:
        # `chunk` is an `AIMessageChunk`; its .content attribute holds the new text.
        txt = chunk.content
        print(txt, end="", flush=True)
        collected += txt
    print("\n--- end of stream ---")

    # 4️⃣ Once the whole string is gathered, parse it with the pydantic parser
    return pydantic_parser.invoke(collected)

# Run it
movie_info = stream_and_parse("What is Iron Man's first movie called?")
print("\n=== Parsed result after streaming ===")
print(movie_info)
